# 03 Gold KPIs
Builds Gold KPI tables for dashboards.

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "health_cat", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("gold_schema", "gold", "Gold schema")
dbutils.widgets.text("write_mode", "overwrite", "Write mode")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
write_mode = dbutils.widgets.get("write_mode")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")
appointments = spark.table(f"{catalog}.{silver_schema}.appointments_enriched")
kaggle = spark.table(f"{catalog}.{silver_schema}.kaggle_noshow_clean")
diagnostics = spark.table(f"{catalog}.{bronze_schema}.diagnostics")
feedback = spark.table(f"{catalog}.{bronze_schema}.feedback")

def write_delta_table(df, table_name):
    writer = df.write.mode(write_mode).format("delta")
    if write_mode == "overwrite":
        writer = writer.option("overwriteSchema", "true")
    writer.saveAsTable(f"{catalog}.{gold_schema}.{table_name}")

In [0]:
no_show_rate = (
    appointments.groupBy("department_name")
    .agg(
        F.count("appointment_id").alias("total_appointments"),
        F.sum("no_show_int").alias("no_show_count")
    )
    .withColumn("no_show_rate_pct", F.round(F.col("no_show_count") / F.col("total_appointments") * 100, 2))
)
write_delta_table(no_show_rate, "gold_no_show_rate")

In [0]:
doctor_utilization = (
    appointments.groupBy("doctor_id", "doctor_name", "department_name")
    .agg(
        F.count("appointment_id").alias("total_appointments"),
        F.sum(F.when(F.col("appointment_status_clean") == "Completed", 1).otherwise(0)).alias("completed_appointments"),
        F.sum(F.when(F.col("appointment_status_clean") == "No Show", 1).otherwise(0)).alias("no_show_appointments")
    )
)
write_delta_table(doctor_utilization, "gold_doctor_utilization")

In [0]:
department_revenue = (
    appointments.groupBy("department_name")
    .agg(F.round(F.sum("bill_amount"), 2).alias("total_revenue"))
    .fillna({"total_revenue": 0})
)
write_delta_table(department_revenue, "gold_department_revenue")

In [0]:
wait_time_trends = (
    appointments.groupBy("visit_month", "department_name")
    .agg(F.round(F.avg("wait_time_minutes"), 2).alias("avg_wait_time_minutes"))
)
write_delta_table(wait_time_trends, "gold_wait_time_trends")

In [0]:
patient_revisit = (
    appointments.groupBy("patient_id")
    .agg(F.count("appointment_id").alias("visit_count"))
    .withColumn("is_revisit_patient", F.col("visit_count") > 1)
)
write_delta_table(patient_revisit, "gold_patient_revisit_rate")

In [0]:
diagnostics_volume = (
    diagnostics.groupBy("test_category", "test_name")
    .agg(F.count("diagnostic_id").alias("test_count"))
)
write_delta_table(diagnostics_volume, "gold_diagnostics_volume")

feedback_summary = (
    feedback.agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("feedback_id").alias("feedback_count")
    )
)
write_delta_table(feedback_summary, "gold_feedback_summary")

In [0]:
kaggle_no_show_by_age = (
    kaggle.withColumn("age_group", F.when(F.col("Age") < 18, "0-17").when(F.col("Age") < 35, "18-34").when(F.col("Age") < 55, "35-54").otherwise("55+"))
    .groupBy("age_group", "Gender")
    .agg(F.count("appointment_id").alias("appointments"), F.sum("no_show_int").alias("no_shows"))
    .withColumn("no_show_rate_pct", F.round(F.col("no_shows") / F.col("appointments") * 100, 2))
)
write_delta_table(kaggle_no_show_by_age, "gold_kaggle_no_show_by_age")

In [0]:
for table in ["gold_no_show_rate", "gold_doctor_utilization", "gold_department_revenue", "gold_wait_time_trends", "gold_patient_revisit_rate", "gold_diagnostics_volume", "gold_feedback_summary", "gold_kaggle_no_show_by_age"]:
    print(table)
    display(spark.table(f"{catalog}.{gold_schema}.{table}"))


gold_no_show_rate


department_name,total_appointments,no_show_count,no_show_rate_pct
Orthopedics,1,0,0.0
Pediatrics,2,0,0.0
Cardiology,3,1,33.33
Dermatology,1,0,0.0
Neurology,1,1,100.0


gold_doctor_utilization


doctor_id,doctor_name,department_name,total_appointments,completed_appointments,no_show_appointments
6,Dr. Arjun Rao,Neurology,1,0,1
3,Dr. Priya Nair,Orthopedics,1,1,0
1,Dr. Ananya Sharma,Cardiology,2,2,0
5,Dr. Neha Iyer,Pediatrics,2,1,0
2,Dr. Rohan Mehta,Cardiology,1,0,1
4,Dr. Karan Malhotra,Dermatology,1,1,0


gold_department_revenue


...

test_category,test_name,test_count
Radiology,X-Ray Knee,1
Cardiology,ECG,1
Blood Test,Lipid Profile,1
Vitals,Blood Pressure Monitoring,1


gold_feedback_summary


avg_rating,feedback_count
4.2,5


gold_kaggle_no_show_by_age


age_group,Gender,appointments,no_shows,no_show_rate_pct
55+,F,19491,3131,16.06
0-17,M,13481,2935,21.77
18-34,M,6482,1542,23.79
55+,M,9438,1429,15.14
0-17,F,13899,3062,22.03
18-34,F,17764,4272,24.05
35-54,F,20686,4129,19.96
35-54,M,9286,1819,19.59
